In [ ]:
# OPTIONAL: Install
# pip install -qU langchain langchain-community langchain-openai duckduckgo-search numexpr pydantic python-dotenv


## Tutorial: Designing a Multi‑Tool Chatbot w/ Functions
In this lesson, we’ll build a tool‑using assistant with function calling and routing logic.



In [ ]:
from getpass import getpass
from langchain_openai import ChatOpenAI

# Enter your OpenRouter API key securely when prompted.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

# OpenRouter provides an OpenAI-compatible API.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# You can change this to any compatible OpenRouter model.
MODEL = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    model=MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    seed=42,
)

print("OpenRouter configured successfully.")
print("Model:", MODEL)
from langchain.tools import tool, Tool
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, seed=42)


### Step 1: Define tools + function schemas
We’ll provide a calculator and a weather tool with strict args.


In [ ]:
import numexpr as ne
from typing import Literal

class CalcRequest(BaseModel):
    expression: str = Field(..., description="Math expression, e.g., '(12+8)/5'")

@tool("calculator", args_schema=CalcRequest, return_direct=True)
def calculator(expression: str) -> str:
    """
    Evaluate simple math expressions, return a number as string.
    """
    try:
        value = ne.evaluate(expression)
        return str(value.item())
    except Exception as e:
        return f"Error: {e}"

# Backward-compatible alias
calc_tool = calculator

class WeatherRequest(BaseModel):
    city: str = Field(..., description="City name, e.g., 'San Francisco'")
    unit: Literal["celsius", "fahrenheit"] = Field("celsius", description="Temperature unit")

@tool("get_weather", args_schema=WeatherRequest, return_direct=True)
def get_weather(city: str, unit: str = "celsius") -> str:
    """
    Get the weather in a given city.
    """
    sample = {"San Francisco": 18, "New York": 24, "London": 19}
    temp_c = sample.get(city, 20)
    if unit == "fahrenheit":
        temp = round((temp_c * 9/5) + 32)
        return f'{{"city": "{city}", "temp": {temp}, "unit": "F"}}'
    return f'{{"city": "{city}", "temp": {temp_c}, "unit": "C"}}'


### Step 2: Create a function‑calling agent
We’ll let the model decide when to call a tool vs answer directly.


In [ ]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor, create_tool_calling_agent

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools when needed. Be concise."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])


In [ ]:
agent = create_tool_calling_agent(llm, tools=[calc_tool, get_weather], prompt=prompt)
executor = AgentExecutor(agent=agent, tools=[calc_tool, get_weather], verbose=True)

print(executor.invoke({"input": "What is (12+8)/5?"})["output"])
print(executor.invoke({"input": "Weather in London in fahrenheit"})["output"])